# AI Agent for Geospatial Resource Discovery

This notebook creates a "Code Agent" that discovers open-source geospatial resources. By providing a topic, the agent will search for relevant datasets, notebooks, and publications. It then uses a large language model to evaluate their quality and relevance and organizes the metadata into a clear, structured format.

## 1. Environment Setup

First, we'll install the necessary libraries and configure our environment.

In [1]:
# Cell 1: Install Libraries
!pip install smolagents duckduckgo-search -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


#### **Set Up Hugging Face API Token**

The agent uses a powerful language model from the Hugging Face Hub. To access it, you need an API token.

1.  If you don't have one, create a free Hugging Face account and get a token from your [**profile settings**](https://huggingface.co/settings/tokens).
2.  Run the cell below. A text field will appear.
3.  Paste your token into the field and press Enter. The token will be hidden for security.

This will load the token and initialize the agent's brain (the language model).

In [2]:
# Cell 2: Import Libraries and Set API Token
import os
import pandas as pd
from smolagents import CodeAgent, tool
from smolagents.models import InferenceClientModel
from duckduckgo_search import DDGS
import getpass

# Prompt for the Hugging Face API token
hf_token = getpass.getpass('🔑 Please enter your Hugging Face API token: ')
os.environ['HF_TOKEN'] = hf_token

if hf_token:
    print("✅ Hugging Face API token has been set.")
else:
    print("⚠️ Hugging Face API token was not provided.")

# Initialize the model that will power our agent
model = InferenceClientModel()

print("🤖 Model initialized.")

🔑 Please enter your Hugging Face API token:  ········


✅ Hugging Face API token has been set.
🤖 Model initialized.


## 2. Create the Agent's "Skills" (Tools)

Next, we define the functions our agent can use. Each function is a "tool" decorated with `@tool`. The agent reads the function's description (the docstring) to understand how to use it.

In [7]:
# Cell 3: Define Geospatial Resource Discovery Tools

@tool
def search_geospatial_resources(topic: str, resource_type: str) -> list:
    """
    Searches for geospatial resources like datasets, notebooks, or publications on a given topic.
    
    Args:
        topic: The geospatial topic to search for.
        resource_type: The type of resource to find ('datasets', 'notebooks', or 'publications').
        
    Returns:
        A list of search results with titles, links, and snippets.
    """
    query_map = {
        'datasets': f'geospatial open data {topic}',
        'notebooks': f'jupyter notebook {topic} github',
        'publications': f'research paper {topic} pdf'
    }
    query = query_map.get(resource_type, f'geospatial {topic}')
    
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=5))
    return results

@tool
def analyze_and_organize_results(results: list, topic: str) -> str:
    """
    Analyzes a list of search results using an LLM to determine quality and relevance.
    It then organizes the extracted metadata into a Markdown table.
    
    Args:
        results: A list of search results from the search_geospatial_resources tool.
        topic: The original geospatial topic for context.
        
    Returns:
        A Markdown formatted string with the organized metadata.
    """
    if not results:
        return "No results to analyze."
    
    analysis_prompt = f"""
    Analyze the following search results about '{topic}'. For each result, determine its relevance and quality.
    Extract the title, a brief summary, and the direct URL. Format the output as a clean Markdown table
    with columns: 'Title', 'Summary', and 'URL'.
    
    Results:\n{results}
    """
    
    # This is a simplified call to the model. In a real scenario, you might have a more complex
    # function to interact with the LLM and parse its output robustly.
    response = model.generate(analysis_prompt)
    
    return response

print("✅ Agent's resource discovery tools are defined.")

✅ Agent's resource discovery tools are defined.


## 3. Build and Run the Agent

With our tools defined, we can now assemble and run the agent.

In [8]:
# Cell 4: Create and Run the Geospatial Agent
geospatial_agent = CodeAgent(
    tools=[
        search_geospatial_resources,
        analyze_and_organize_results,
    ],
    model=model,
)

print("🤖 Geospatial Agent is built and ready for commands!")

🤖 Geospatial Agent is built and ready for commands!


## 4. Discover Geospatial Resources

Now, let's give our agent a task. We'll ask it to find resources on a specific topic. The agent will plan the search, execute it, analyze the results, and present them in an organized table.

In [9]:
# Cell 5: Main Discovery Task
geospatial_agent.run(
    "Find open-source datasets and research papers about crime cases in Chicago. "
    "Analyze the results and present them in a table."
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find open-source datasets and research papers about crime cases in Chicago. Analyze the results and present     │
│ them in a table.                                                                                                │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  topic = "crime cases in Chicago"                                                                                 
  results_datasets = search_geospatial_resources(topic=topic, resource_type='datasets')                            
  results_publications = search_geospatial_resources(topic=topic, resource_type='publications')                    
                                                                                                                   
  print("Datasets results:", results_datasets)                                                                     
  print("Publications results:", results_publications)                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

/tmp/ipykernel_446/260336623.py:22: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/tmp/ipykernel_446/260336623.py:22: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Execution logs:
Datasets results: [{'title': 'Data | CLEARMAP - Chicago Police Department', 'href': 
'https://gis.chicagopolice.org/pages/data', 'body': 'The information retrieved using CLEARMAP reflects incidents 
reported in Chicago, Illinois. This data reflects incidents where the police responded and completed case reports. 
It should be noted that though the police responded to an incident, a case report may not be generated. Also, this 
information does not reflect 911 calls for service.'}, {'title': 'City of Chicago Crime Data | City of Chicago | 
Data Portal', 'href': 'https://data.cityofchicago.org/Public-Safety/City-of-Chicago-Crime-Data/v9q9-3dm2', 'body': 
'This dataset reflects reported incidents of crime (with the exception of murders where data exists for each 
victim) that occurred in the City of …'}, {'title': 'Data - clearmap-chicagopd.hub.arcgis.com', 'href': 
'https://clearmap-chicagopd.hub.arcgis.com/datasets/data/about', 'body': 'Data Search Chicago Police Department GIS
Data About Data The information retrieved using CLEARMAP reflects incidents reported in …'}, {'title': 'Crimes - 
City of Chicago', 'href': 'https://www.chicago.gov/city/en/dataset/crime.html', 'body': 'This data reflects 
reported incidents of crime that have occurred in the City of Chicago during a specific time period. Data is 
extracted from …'}, {'title': 'Chicago Crime - ArcGIS', 'href': 
'https://experience.arcgis.com/experience/eb15ed73f8c74cee95dde46cfba3147a/page/Crime-Site-Information/', 'body': 
'Use this application to view crime by geographies like CPD District, CPD Beat, Ward and Community Area. Visualize 
how those …'}]
Publications results: [{'title': 'Research - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Research', 'body': 
'Approaches to research depend on epistemologies, which vary considerably both within and between humanities and 
sciences. …'}, {'title': 'ResearchGate | Find and share research', 'href': 'https://www.researchgate.net/', 'body':
'Access 160+ million publication pages and connect with 25+ million researchers. Join for free and gain visibility 
by uploading your …'}, {'title': 'Research | SPJ', 'href': 'https://spj.science.org/journal/research', 'body': '2 
days ago · The Open Access journal Research, published in association with CAST, publishes innovative, wide-ranging
research in life …'}, {'title': 'Google Scholar', 'href': 'https://scholar.google.com/', 'body': 'Google Scholar 
provides a simple way to broadly search for scholarly literature. Search across a wide variety of disciplines and 
sources: …'}, {'title': 'RESEARCH Definition & Meaning - Merriam-Webster', 'href': 
'https://www.merriam-webster.com/dictionary/research', 'body': 'The meaning of RESEARCH is studious inquiry or 
examination; especially : investigation or experimentation aimed at the discovery and …'}]

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(combined_markdown)                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: # Open-Source Resources for Crime Cases in Chicago

## Research Papers
| Title | URL |
|-------|-----|
| Chicago Crime Prediction is a machine learning project focused on analyzing and forecasting crime patterns in the
city of Chicago. | 
[https://github.com/saikiransalama/chicago-crime-prediction/blob/main/Research+Paper.pdf](https://github.com/saikir
ansalama/chicago-crime-prediction/blob/main/Research+Paper.pdf) |
| The Windy City’s Dark Side: A Statistical Exploration of Crime in the City of Chicago | 
[https://www.researchgate.net/publication/381965298_The_Windy_City%27s_Dark_Side_A_Statistical_Exploration_of_Crime
_in_the_City_of_Chicago](https://www.researchgate.net/publication/381965298_The_Windy_City%27s_Dark_Side_A_Statisti
cal_Exploration_of_Crime_in_the_City_of_Chicago) |
| Crime Analysis in Chicago City | 
[https://www.researchgate.net/publication/335361962_Crime_Analysis_in_Chicago_City](https://www.researchgate.net/pu
blication/335361962_Crime_Analysis_in_Chicago_City) |
| Research Paper - Crime and Punishment | 
[https://www.academia.edu/26030017/Research_Paper_Crime_and_Punishment](https://www.academia.edu/26030017/Research_
Paper_Crime_and_Punishment) |
| Artificial intelligence & crime prediction: A systematic literature review of AI techniques and datasets for 
crime prediction | 
[https://www.sciencedirect.com/science/article/pii/S2590291122000961](https://www.sciencedirect.com/science/article
/pii/S2590291122000961) |

## Datasets
| Title | Description | URL |
|-------|-------------|-----|
| City of Chicago Crime Data | City of Chicago | Data Portal | Dataset reflecting reported incidents of crime that 
occurred in the City of Chicago during a specific time period. | 
[https://data.cityofchicago.org/Public-Safety/City-of-Chicago-Crime-Data/v9q9-3dm2](https://data.cityofchicago.org/
Public-Safety/City-of-Chicago-Crime-Data/v9q9-3dm2) |

[Step 5: Duration 1.06 seconds| Input tokens: 20,149 | Output tokens: 1,270]

'# Open-Source Resources for Crime Cases in Chicago\n\n## Research Papers\n| Title | URL |\n|-------|-----|\n| Chicago Crime Prediction is a machine learning project focused on analyzing and forecasting crime patterns in the city of Chicago. | [https://github.com/saikiransalama/chicago-crime-prediction/blob/main/Research+Paper.pdf](https://github.com/saikiransalama/chicago-crime-prediction/blob/main/Research+Paper.pdf) |\n| The Windy City’s Dark Side: A Statistical Exploration of Crime in the City of Chicago | [https://www.researchgate.net/publication/381965298_The_Windy_City%27s_Dark_Side_A_Statistical_Exploration_of_Crime_in_the_City_of_Chicago](https://www.researchgate.net/publication/381965298_The_Windy_City%27s_Dark_Side_A_Statistical_Exploration_of_Crime_in_the_City_of_Chicago) |\n| Crime Analysis in Chicago City | [https://www.researchgate.net/publication/335361962_Crime_Analysis_in_Chicago_City](https://www.researchgate.net/publication/335361962_Crime_Analysis_in_Chicago_City) 